# 23.6 — Markov / stochastic games

A stochastic game is a Markov decision process whose transition and reward depend on everyone’s actions. Save a copy to Drive to edit.

States, joint actions, transitions, rewards, and discounting interact: a local stage-game best response changes the long-run value through future state occupancy. The lesson's warning is to avoid replacing this strategic backup with a single-agent MDP max.

In [ ]:
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(23)

np.set_printoptions(precision=3, suppress=True)


## The concept, built once: fixed-policy Bellman evaluation

For a fixed joint policy, a player's values solve $V=r+\gamma PV$, so $V=(I-\gamma P)^{-1}r$. The lesson's first backup with $r=(1,0)$, $P=egin{bmatrix}0.8&0.2\0.3&0.7\end{bmatrix}$, and $\gamma=0.9$ starts from $V=(0,0)$ and gives $(1,0)$.

In [ ]:
def bellman_backup(P, r, gamma, V):
    return r + gamma * P @ V


def policy_evaluation(P, r, gamma):
    eye = np.eye(P.shape[0])
    return np.linalg.solve(eye - gamma * P, r)


P_lesson = np.array([
    [0.8, 0.2],
    [0.3, 0.7],
])
r_lesson = np.array([1.0, 0.0])
first_backup = bellman_backup(P_lesson, r_lesson, 0.9, np.zeros(2))
V_lesson = policy_evaluation(P_lesson, r_lesson, 0.9)
V_low = policy_evaluation(P_lesson, r_lesson, 0.1)
V_high = policy_evaluation(P_lesson, r_lesson, 0.99)

assert np.allclose(first_backup, np.array([1.0, 0.0]))
assert round(float(V_lesson[0]), 3) == 6.727
assert round(float(V_lesson[1]), 3) == 4.909
assert round(float(V_low[0]), 6) == 1.087719
assert round(float(V_high[0]), 6) == 60.792079

print("First backup:", first_backup)
print("Converged value at gamma=0.9:", V_lesson)
print("Discount sensitivity:", V_low[0], V_lesson[0], V_high[0])


## Strategic coupling inside a state

A stage-game best response evaluates a row action against an opponent policy. For $A=egin{bmatrix}2&0\0&1\end{bmatrix}$ and opponent Left probability $q=0.25$, the action values are $0.5$ and $0.75$, so action 1 is the best response.

In [ ]:
def stage_best_response(A, q):
    opponent_policy = np.array([q, 1.0 - q])
    values = A @ opponent_policy
    best_value = float(np.max(values))
    best_actions = tuple(np.flatnonzero(np.isclose(values, best_value)).tolist())
    return best_actions, values


def joint_policy_transition_reward(P_joint, R_joint, row_policy, col_policy):
    states = P_joint.shape[0]
    P_pi = np.zeros((states, states))
    r_pi = np.zeros(states)
    for s in range(states):
        for a in range(P_joint.shape[1]):
            for b in range(P_joint.shape[2]):
                weight = row_policy[s, a] * col_policy[s, b]
                P_pi[s] += weight * P_joint[s, a, b]
                r_pi[s] += weight * R_joint[s, a, b]

    return P_pi, r_pi


A_stage = np.array([
    [2.0, 0.0],
    [0.0, 1.0],
])
best_actions, action_values = stage_best_response(A_stage, 0.25)

assert round(float(action_values[0]), 6) == 0.5
assert round(float(action_values[1]), 6) == 0.75
assert best_actions == (1,)

print("Stage action values:", action_values)
print("Best response action:", best_actions)


## Dataset ladder: D1–D5 stochastic games of rising coupling

D1 is the fixed-policy lesson chain. D2 adds two actions per player, D3 changes the opponent policy, D4 expands state and joint-action counts, and D5 uses high discounting plus transition control so MDP-style maximization becomes misleading.

In [ ]:
def random_stochastic_game(states, actions, seed, coupling):
    local_rng = np.random.default_rng(seed)
    P_joint = local_rng.random((states, actions, actions, states))
    P_joint = P_joint / P_joint.sum(axis=-1, keepdims=True)
    base_reward = local_rng.normal(0.0, 0.4, size=(states, actions, actions))
    diagonal_bonus = np.zeros_like(base_reward)
    for a in range(actions):
        diagonal_bonus[:, a, a] = coupling
    R_joint = base_reward + diagonal_bonus
    row_policy = np.full((states, actions), 1.0 / actions)
    col_policy = np.full((states, actions), 1.0 / actions)
    return P_joint, R_joint, row_policy, col_policy


def build_stochastic_ladder():
    d1 = {
        "name": "D1 fixed-policy lesson chain",
        "P": P_lesson,
        "r": r_lesson,
        "gamma": 0.9,
        "stage": A_stage,
        "q": 0.25,
    }
    P2, R2, row2, col2 = random_stochastic_game(2, 2, 2362, 1.0)
    d2 = {
        "name": "D2 two-state two-action game",
        "P_joint": P2,
        "R_joint": R2,
        "row_policy": row2,
        "col_policy": col2,
        "gamma": 0.85,
    }
    P3, R3, row3, col3 = random_stochastic_game(3, 2, 2363, 0.7)
    col3[:, 0] = np.array([0.8, 0.2, 0.6])
    col3[:, 1] = 1.0 - col3[:, 0]
    d3 = {
        "name": "D3 nonstationary opponent policy",
        "P_joint": P3,
        "R_joint": R3,
        "row_policy": row3,
        "col_policy": col3,
        "gamma": 0.9,
    }
    P4, R4, row4, col4 = random_stochastic_game(5, 3, 2364, 0.8)
    d4 = {
        "name": "D4 many states and joint actions",
        "P_joint": P4,
        "R_joint": R4,
        "row_policy": row4,
        "col_policy": col4,
        "gamma": 0.92,
    }
    P5, R5, row5, col5 = random_stochastic_game(6, 3, 2365, 1.2)
    for s in range(6):
        P5[s, 2, 0] = np.eye(6)[(s + 1) % 6]
        R5[s, 2, 0] += 1.5
    col5[:, 0] = 0.7
    col5[:, 1] = 0.2
    col5[:, 2] = 0.1
    d5 = {
        "name": "D5 high-gamma strategic coupling",
        "P_joint": P5,
        "R_joint": R5,
        "row_policy": row5,
        "col_policy": col5,
        "gamma": 0.98,
    }
    return [d1, d2, d3, d4, d5]


stochastic_ladder = build_stochastic_ladder()
for rung in stochastic_ladder:
    print(rung["name"])
    if "P" in rung:
        print("  states:", rung["P"].shape[0], "fixed policy")
        print("  P sample:", rung["P"])
    else:
        print("  states/actions:", rung["P_joint"].shape[0], rung["P_joint"].shape[1])
        print("  reward sample state 0:", np.round(rung["R_joint"][0], 3))


In [ ]:
def row_best_response_regret(P_joint, R_joint, row_policy, col_policy, gamma):
    P_pi, r_pi = joint_policy_transition_reward(P_joint, R_joint, row_policy, col_policy)
    V = policy_evaluation(P_pi, r_pi, gamma)
    regrets = []
    for s in range(P_joint.shape[0]):
        q_values = []
        current_value = 0.0
        for a in range(P_joint.shape[1]):
            expected = 0.0
            for b in range(P_joint.shape[2]):
                expected += col_policy[s, b] * (R_joint[s, a, b] + gamma * P_joint[s, a, b] @ V)
            q_values.append(expected)
            current_value += row_policy[s, a] * expected
        regrets.append(max(q_values) - current_value)

    return V, float(np.max(regrets)), np.array(regrets)


stochastic_results = []
for rung in stochastic_ladder:
    if "P" in rung:
        V = policy_evaluation(rung["P"], rung["r"], rung["gamma"])
        best_actions, stage_values = stage_best_response(rung["stage"], rung["q"])
        regret = float(np.max(stage_values) - np.mean(stage_values))
        states = rung["P"].shape[0]
    else:
        V, regret, state_regrets = row_best_response_regret(
            rung["P_joint"],
            rung["R_joint"],
            rung["row_policy"],
            rung["col_policy"],
            rung["gamma"],
        )
        states = rung["P_joint"].shape[0]
    stochastic_results.append({
        "name": rung["name"],
        "states": states,
        "gamma": rung["gamma"],
        "value0": float(V[0]),
        "regret": regret,
        "values": V,
    })

print("rung | states | gamma | V(s0) | best-response regret")
for result in stochastic_results:
    print(result["name"], result["states"], result["gamma"], round(result["value0"], 3), round(result["regret"], 3))


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for col, rung in enumerate(stochastic_ladder):
    ax = axes[col]
    if "P" in rung:
        matrix = rung["P"]
        title = "D1 P"
    else:
        matrix = rung["P_joint"][:, 0, 0, :]
        title = rung["name"].split()[0] + " P[a0,b0]"
    image = ax.imshow(matrix, cmap="viridis", vmin=0)
    ax.set_title(title)
    ax.set_xlabel("next state")
    ax.set_ylabel("state")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white")

fig.colorbar(image, ax=axes[-1], shrink=0.8)
fig.tight_layout()
plt.show()

sizes = [result["states"] for result in stochastic_results]
values = [result["value0"] for result in stochastic_results]
regrets = [result["regret"] for result in stochastic_results]

plt.figure(figsize=(7, 4))
plt.plot(sizes, values, marker="o", label="V(s0)")
plt.plot(sizes, regrets, marker="s", label="best-response regret")
plt.xlabel("number of states")
plt.ylabel("metric")
plt.title("Value/regret vs. stochastic-game size")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Pitfall on D5: using an MDP max where a stage game is needed

A single-agent backup maximizes over the row action as if the opponent disappeared. The corrected backup evaluates row actions against the opponent's policy and future values, then reports best-response regret rather than pretending the MDP optimum is an equilibrium.

In [ ]:
d5 = stochastic_ladder[-1]
P_pi, r_pi = joint_policy_transition_reward(d5["P_joint"], d5["R_joint"], d5["row_policy"], d5["col_policy"])
V_joint = policy_evaluation(P_pi, r_pi, d5["gamma"])
wrong_rewards = d5["R_joint"].max(axis=(1, 2))
wrong_P = d5["P_joint"][np.arange(d5["P_joint"].shape[0]), 2, 0]
wrong_V = policy_evaluation(wrong_P, wrong_rewards, d5["gamma"])
fixed_V, fixed_regret, state_regrets = row_best_response_regret(
    d5["P_joint"],
    d5["R_joint"],
    d5["row_policy"],
    d5["col_policy"],
    d5["gamma"],
)

print("Wrong MDP-style V(s0):", round(float(wrong_V[0]), 3))
print("Joint-policy V(s0):", round(float(V_joint[0]), 3))
print("Fixed best-response regret:", round(fixed_regret, 3))
print("State regrets:", np.round(state_regrets, 3))


## Evaluate it + Practice

- Metric: value error for fixed-policy evaluation and best-response regret for strategic rungs; compare with a no-skill baseline that keeps uniform random policies.
- Cheap sanity check: every transition distribution must sum to one.
- Ablation: lower $\gamma$ on D5 and verify long-run transition control matters less.
- Failure signals: using max over joint actions for one player, treating opponent policies as stationary after changing them, or reporting only immediate rewards.

Practice prompt 1: Change D2's opponent policy and compute the new row best-response regret.

Practice prompt 2: Sweep $\gamma$ for D1 and plot $V(s_0)$ against patience.

Practice prompt 3: Replace D5's controlled transition with a random one and compare regret.